# 02 · Preparing MaleCNS data: from a table of synapses to a brain

**Goal:** see exactly how the repo goes from the raw tables (notebook 01) to the matrix **W** that the fly's brain runs on.

You will learn
- **groups** (`fly/groups.yaml`): named sets of neurons picked by cell type and side
- the preparation steps: pick a cast → keep strong connections → apply **Dale's law** → normalise each neuron's inputs → a weight matrix **W**
- how the repo's real brain (`fly/brain.py`) differs from our hand-built mini version, and why
- how **mechanisms** add extra neurons (hearing, smell) to the base brain

> Real vs invented: the neurons, the synapse counts and the signs are **teal** (data). Choosing *which* neurons form the cast, the "≥ 5 synapses" threshold and the row normalisation are **orange** (our modelling choices).

## 0. Data

Set the folder with the MaleCNS files (see `tutorials/README.md`).

In [ ]:
DATA_DIR = "~/fly_mario_fly_data"      # <-- change this to the folder that holds malecns_v1.0/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import _helpers as H
H.use_data_folder(DATA_DIR)

from fly_mario_fly.fly.connectome import Connectome
conn = Connectome()
print(len(conn.ids), "neurons loaded")

## 1. Groups: the repo's vocabulary

`src/fly_mario_fly/fly/groups.yaml` is **the one table** of neuron groups. A group = cell types (+ optional side) + a role in plain words + a colour. Senses write to groups,
actions read groups, renderers colour by group. Nothing else needs to know individual neurons.

In [ ]:
from fly_mario_fly.fly.groups import load_groups, masks_for
groups = load_groups()
rows = [(name, g.types or f"prefix {g.type_prefix}", g.side or "both", g.role) for name, g in groups.items()]
pd.set_option("display.max_colwidth", 80)
pd.DataFrame(rows, columns=["group", "types", "side", "role"])

In [ ]:
masks = masks_for(conn.type, conn.soma_side)      # one boolean mask over ALL 211k neurons per group
print({name: int(m.sum()) for name, m in masks.items() if name in ("LC4_L", "LC4_R", "LPLC2_L", "LPLC2_R", "DNp01", "TTMn", "DNp09")})

## 2. A hand-built mini brain: the escape circuit

The repo's *escape* mechanism (`fly/mechanisms/escape.py`) is: looming detectors **LC4 / LPLC2** → Giant Fiber **DNp01** → jump muscle motor neuron **TTMn**.
Let's build it ourselves from the raw tables with just those four types (315 neurons).

**Step 1 — pick the cast.**

In [ ]:
cast_types = ["LC4", "LPLC2", "DNp01", "TTMn"]
cast = conn.body_ids(types=cast_types)
rows_of = conn.index(cast)                      # row of each cast neuron in the annotation arrays
print(len(cast), "neurons:", {t: int((conn.type[rows_of] == t).sum()) for t in cast_types})

**Step 2 — keep strong connections.** `conn.edge_table()` makes one streaming pass over the 150 M-row file, keeps every connection of **≥ 5 synapses** between annotated neurons
(that's the project's *N = 5 rule*), and **caches** the result in `$FLY_MARIO_FLY_DATA/derived/`. The first call takes a minute or two; afterwards it takes seconds.

In [ ]:
import time
t0 = time.time()
pre, post, w = conn.edge_table()                # (rows of conn.ids)
print(f"{len(pre):,} connections with >= 5 synapses   ({time.time() - t0:.0f} s)")

# keep connections whose two ends are both in the cast
local = {int(r): k for k, r in enumerate(rows_of)}      # annotation row -> position in the cast
keep = np.isin(pre, rows_of) & np.isin(post, rows_of)
a = np.array([local[int(r)] for r in pre[keep]])         # pre-synaptic position in the cast
b = np.array([local[int(r)] for r in post[keep]])        # post-synaptic position in the cast
counts = np.zeros((len(cast), len(cast)))
np.add.at(counts, (b, a), w[keep])                       # counts[post, pre] = synapses
print("connections among the cast:", keep.sum(), "| synapses:", int(counts.sum()))

**Step 3 — signs (Dale's law)** from the transmitter, as in notebook 01. **Step 4 — normalise:** each neuron's inputs are divided by its total input, so a row of **W** says *"what fraction of my input comes from each neuron"*, then multiplied by that source neuron's sign:

$$W_{ij} = \mathrm{sign}_j \cdot \frac{\text{synapses } j \to i}{\sum_k \text{synapses } k \to i}$$

This is `_finish()` in `fly/brain.py`.

In [ ]:
signs = conn.signs(cast)
row_sum = counts.sum(axis=1, keepdims=True)
W_mini = np.divide(counts, row_sum, out=np.zeros_like(counts), where=row_sum > 0) * signs[np.newaxis, :]
print("W:", W_mini.shape, "| non-zero:", f"{(W_mini != 0).mean():.1%}", "| neurons with no input in this cast:", int((row_sum == 0).sum()))

In [ ]:
# order the cast by type so the blocks are visible
order = np.argsort(conn.type[rows_of], kind="stable")
labels = conn.type[rows_of][order]
fig, ax = plt.subplots(figsize=(6.2, 5.6))
im = ax.imshow(W_mini[np.ix_(order, order)], cmap="RdBu_r", vmin=-0.2, vmax=0.2)
for t in cast_types:
    idx = np.flatnonzero(labels == t)
    ax.axhline(idx[0] - 0.5, color="k", lw=0.4); ax.axvline(idx[0] - 0.5, color="k", lw=0.4)
    ax.text(idx.mean(), -6, t, ha="center", fontsize=8)
ax.set_xlabel("from (pre-synaptic)"); ax.set_ylabel("to (post-synaptic)")
plt.colorbar(im, label="weight (red = excitatory, blue = inhibitory)"); plt.show()

Read it as: **column = sender, row = receiver**; the neurons are sorted by type (DNp01, LC4, LPLC2, TTMn). What the picture shows:

- two bright diagonal blocks: each detector pool (**LC4**, **LPLC2**) is densely connected **to itself** (recurrence), while the two pools barely talk to each other (the empty off-diagonal blocks);
- the top rows (**DNp01**, the Giant Fiber) receive from many LC4 and LPLC2 neurons;
- the single dot in the bottom-left corner is DNp01 → **TTMn**, the jump motor neuron.

That recurrence inside the detector pools is why the brain is run as a *dynamical system* in notebook 04, not as a one-step lookup.

In [ ]:
dn = np.flatnonzero(conn.type[rows_of] == "DNp01")
print("inputs of the two Giant Fibers (DNp01), by sending type:")
for i in dn:
    frac = pd.Series(np.abs(W_mini[i]), index=conn.type[rows_of]).groupby(level=0).sum()
    print("  ", int(cast[i]), frac[frac > 0].round(3).to_dict())

## 3. The repo's real brain

Our mini brain has 315 neurons and *only* the escape cast. The repo's base brain, `steer636`, has **636**: the escape circuit, the walking neurons (DNp09), the steering neurons (LC10a → DNa) plus 40 **relay** neurons.
Differences from our hand-built version, written in `fly/brain.py`:

- it keeps **every** synapse count among the cast (`conn.edges_among`) rather than only connections ≥ 5;
- the extra relay neurons are chosen by a rule (`N_RELAYS = 40`: top bodies by min(synapses from LC10a, synapses to DNa01/02));
- rows are normalised over **all 636** neurons.

It is built once, cached by a content hash of its recipe and the data, and loaded from `$FLY_MARIO_FLY_DATA/brains/`. (If the cache isn't there yet, the next cell builds it: that streams the big file once and takes a couple of minutes.)

In [ ]:
from fly_mario_fly.fly import brain as fb
from fly_mario_fly.fly.spec import BrainSpec

brain = fb.build_brain(BrainSpec())
print("neurons:", brain.n, "| W:", brain.W.shape, "| non-zero:", f"{(brain.W != 0).mean():.1%}")
print("group sizes:", brain.counts())
print("recipe name:", fb.brain_name(BrainSpec()), "| cache file:", fb.brain_path(BrainSpec()).name)

In [ ]:
# sanity check: each row is "fraction of my input" with signs applied, so the absolute values sum to 1 (or 0 if no input)
sums = np.abs(brain.W).sum(axis=1)
print("row sums of |W|: min", sums.min().round(3), "max", sums.max().round(3), "| rows with no input:", int((sums == 0).sum()))

### The brain at group level

Averaging **W** over the neurons of each group gives a wiring diagram you can read. Value = mean weight from a neuron in the column group onto a neuron in the row group.
The bright diagonal is recurrence within a group; the row `DNp01` shows the detectors feeding the Giant Fiber; `DNp09` (walking) listens to `LC10a`/`LPLC2`; `DNa` (steering) listens to `DNp09`.

In [ ]:
names = list(brain.masks)
G = np.array([[brain.W[np.ix_(brain.masks[r], brain.masks[c])].mean() for c in names] for r in names])
print("mean weight TTMn <- DNp01:", G[names.index("TTMn"), names.index("DNp01")].round(2), "(off the colour scale below on purpose: DNp01 supplies almost all of TTMn's input)")
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(G, cmap="RdBu_r", vmin=-0.02, vmax=0.02)
ax.set_xticks(range(len(names)), names, rotation=90, fontsize=8); ax.set_yticks(range(len(names)), names, fontsize=8)
ax.set_xlabel("from"); ax.set_ylabel("to"); plt.colorbar(im, label="mean weight"); plt.show()

## 4. Mechanisms add neurons

Escape and walking live inside the 636 neurons. **Hearing** (JO neurons → Giant Fiber) and **smell** (ORN receptors → steering) need *more* neurons, so each is a **mechanism** plug-in (`fly/mechanisms/*.py`) that appends extra neurons and their connections to the base.
A brain is identified by which mechanisms it has (`BrainSpec(mechanisms=...)`); `all_specs()` lists every useful combination. A fly's attachments decide which mechanisms it needs.

In [ ]:
from fly_mario_fly import registry
for name, m in registry.list_plugins("mechanisms").items():
    print(f"{name:11s} adds neurons: {'yes' if m.extra else 'no ':3s}  {m.label}")
print()
for spec in fb.all_specs():
    p = fb.brain_path(spec)
    print(f"{str(spec.mechanisms):55s} cached: {p.exists()}")
print()
print("(brains you have not built show 'cached: False'; `python scripts/download_data.py --build-brains` builds them all)")

## Recap

- **Groups** (`groups.yaml`) name sets of neurons by type and side; everything else speaks in groups.
- **Preparation** = pick a cast → keep connections (≥ 5 synapses for added neurons) → Dale's-law signs → row-normalise → **W**.
- The brain is **built once and cached** (by a content hash), then loaded.
- Mechanisms add neurons: the *base* never changes, so older flies behave exactly the same.

**Try this**
1. Add `"DNp09"` to `cast_types` and rebuild the mini brain. Does the Giant Fiber connect to it?
2. Change the `>= 5` threshold: call `conn.edge_table(min_syn=3)` (a new cache file!) and see how `W_mini` gets denser.
3. Which group receives input from `LC10a`? Read it off the group-level matrix.

**Next:** notebook 04 runs this brain over time and watches how it reacts.